In [14]:
import requests
import pandas as pd 
import numpy as np
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import mean_squared_error
import pytz
import matplotlib.pyplot as plt

In [51]:
api_key = '32c036998d9b552a254f283f15a6dd62'
base_url = 'https://api.openweathermap.org/data/2.5/'

In [52]:

def get_current_weather(city):
    base_url = "http://api.openweathermap.org/data/2.5/"
    api_key = "32c036998d9b552a254f283f15a6dd62"  
    url = f"{base_url}weather?q={city}&appid={api_key}&units=metric"  
    
    response = requests.get(url)
    data = response.json()
    
   
    if response.status_code == 200:
        return {
            'city': data['name'],
            'current_temp': round(data['main']['temp']),
            'feels_like': round(data['main']['feels_like']),
            'temp_min': round(data['main']['temp_min']),
            'temp_max': round(data['main']['temp_max']),
            'humidity': round(data['main']['humidity']),
            'description': data['weather'][0]['description'],
            'country': data['sys']['country'],
            'wind_gust_dir': data['wind']["deg"],
            'pressure': data['main']['pressure'],
            'wind_gust_speed': data['wind']['speed'],
        }
    else:
        print(f"Error fetching data: {data.get('message', 'Unknown error')}")
        return None


In [85]:
def read_historical_data(weather):
    df = pd.read_csv(weather)
    df = df.dropna()
    df = df.drop_duplicates()
    return df


In [86]:
def prepare_data(data):
    le = LabelEncoder()
    data['WindGustDir'] = le.fit_transform(data['WindGustDir'])
    data['RainTomorrow'] = le.fit_transform(data['RainTomorrow'])

    X = data[['MinTemp', 'MaxTemp', 'WindGustDir', 'WindGustSpeed', 'Humidity', 'Pressure', 'Temp']]
    y = data['RainTomorrow']

    return X, y, le

In [87]:
def train_rain_model(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = RandomForestClassifier(n_estimators=100, random_state=42) 
    model.fit(X_train, y_train)  

    y_pred = model.predict(X_test)
    print("Mean squared error for rain model:")
    print(mean_squared_error(y_test, y_pred))

    return model


In [88]:
def prepare_regression_data(df, target_column, window_size=5):
    data = df[[target_column]].copy()

    features = []
    labels = []

    for i in range(len(data) - window_size):
        feature = data.iloc[i:i+window_size].values.flatten()
        label = data.iloc[i + window_size][target_column]
        features.append(feature)
        labels.append(label)

    return np.array(features), np.array(labels)


    
    

In [89]:
def train_regression_model(X, y):
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X, y)
    return model
                                  

In [90]:
def predict_future(model, current_sequence, steps=5):
    predictions = []
    input_seq = current_sequence.copy()

    for _ in range(steps):
        next_value = model.predict([input_seq])  # predict with the full window
        predictions.append(next_value[0])
        input_seq = input_seq[1:] + [next_value[0]]  # slide window forward

    return predictions



In [92]:
def weather_view():
    city = input('Enter city name: ')
    current_weather = get_current_weather(city)
    
    if current_weather is None:
        print("Error fetching weather data. Exiting.")
        return
    
    
    print("Raw Weather Data:", current_weather)

    
    current_temp_c = current_weather['current_temp']
    min_temp_c = current_weather['temp_min']
    max_temp_c = current_weather['temp_max']
    feels_like_c = current_weather['feels_like']

   
    historical_data = read_historical_data('weather.csv')  
    X, y, le = prepare_data(historical_data)
    rain_model = train_rain_model(X, y)
    # Extract last 5 values for 'Temp' and 'Humidity' from historical data
    last_temp_values = historical_data['Temp'].values[-5:].tolist()
    last_humidity_values = historical_data['Humidity'].values[-5:].tolist()

    future_temp = predict_future(temp_model, last_temp_values)
    future_humidity = predict_future(hum_model, last_humidity_values)


    # wind direction to compass points
    wind_deg = current_weather['wind_gust_dir'] % 360
    compass_points = [
        ("N", 0, 11.25), ("NE", 11.25, 33.75), ("NE", 33.75, 56.25),
        ("ENE", 56.25, 78.75), ("E", 78.75, 101.25), ("ESE", 101.25, 123.75),
        ("SE", 123.75, 146.25), ("SSE", 146.25, 168.75), ("S", 168.75, 191.25),
        ("SSW", 191.25, 213.75), ("SW", 213.75, 236.25), ("WSW", 236.25, 258.75),
        ("W", 258.75, 281.25), ("WNW", 281.25, 303.75), ("NW", 303.75, 326.25),
        ("NNW", 326.25, 348.75)
    ]
    compass_direction = next(
        point for point, start, end in compass_points if start <= wind_deg < end
    )

    compass_direction_encoded = (
        le.transform([compass_direction])[0] if compass_direction in le.classes_ else -1
    )

    current_data = {
        'MinTemp': min_temp_c,
        'MaxTemp': max_temp_c,
        'WindGustDir': compass_direction_encoded,
        'WindGustSpeed': current_weather['wind_gust_speed'],
        'Humidity': current_weather['humidity'],
        'Pressure': current_weather['pressure'],
        'Temp': current_temp_c
    }

    current_df = pd.DataFrame([current_data])
    X_temp, y_temp = prepare_regression_data(historical_data, 'Temp')
    X_hum, y_hum = prepare_regression_data(historical_data, 'Humidity')

    temp_model = train_regression_model(X_temp, y_temp)
    hum_model = train_regression_model(X_hum, y_hum)

    last_temp_values = historical_data['Temp'].values[-5:].tolist()
    last_humidity_values = historical_data['Humidity'].values[-5:].tolist()

    future_temp = predict_future(temp_model, last_temp_values)
    future_humidity = predict_future(hum_model, last_humidity_values) 

    # Rain prediction
    rain_prediction = rain_model.predict(current_df)[0]

    #regression models for temperature and humidity
    X_temp, y_temp = prepare_regression_data(historical_data, 'Temp')
    X_hum, y_hum = prepare_regression_data(historical_data, 'Humidity')

    temp_model = train_regression_model(X_temp, y_temp)
    hum_model = train_regression_model(X_hum, y_hum)

    #predict temperature and humidity
    future_temp = predict_future(temp_model, current_weather['temp_min'])
    future_humidity = predict_future(hum_model, current_weather['humidity'])
    
    
 #defining timezone
    timezone = pytz.timezone('Asia/Karachi')  
    now = datetime.now(timezone)
    next_hour = now + timedelta(hours=1)
    next_hour = next_hour.replace(minute=0, second=0, microsecond=0)
    future_humidity = predict_future_with_hour(hum_model, current_weather['humidity'], now.hour)

    future_times = [(next_hour + timedelta(hours=i)).strftime("%H:00") for i in range(5)]
 
    #outputs
    print(f"City: {city}, Current weather: {current_weather['country']}")
    print(f"Current Temperature: {round(current_temp_c, 1)}°C") 
    print(f"Feels like: {round(feels_like_c, 1)}°C")
    print(f"Minimum Temperature: {round(min_temp_c, 1)}°C")
    print(f"Maximum Temperature: {round(max_temp_c, 1)}°C")
    print(f"Humidity: {current_weather['humidity']}%")
    print(f"Weather Prediction: {current_weather['description']}")
    print(f"Will it rain? : {'Yes' if rain_prediction else 'No'}")

    print("\nFuture Temperature Predictions:")
    for time, temp in zip(future_times, future_temp):
        print(f"{time}: {round(temp, 1)}°C")

    print("\nFuture Humidity Predictions: ")
    for time, humidity in zip(future_times, future_humidity):
        print(f"{time}: {round(humidity, 1)}%")

#PLOT 1: Current Weather 
    labels = ['Current', 'Feels Like', 'Min', 'Max', 'Humidity']
    values = [current_temp_c, feels_like_c, min_temp_c, max_temp_c, current_weather['humidity']]

    plt.figure(figsize=(8, 5))
    plt.bar(labels, values, color='skyblue')
    plt.title(f"{city} - Current Weather Summary")
    plt.ylabel('°C / %')
    plt.grid(axis='y', linestyle='--')
    plt.tight_layout()
    plt.show()


 #PLOT 2: Future Temperature
    plt.figure(figsize=(8, 5))
    plt.plot(future_times, future_temp, marker='o', color='orange')
    plt.title('Future Temperature Predictions')
    plt.xlabel('Time')
    plt.ylabel('Temperature (°C)')
    plt.grid(True)
    plt.tight_layout()
    plt.show()    

 #PLOT 3: Future Humidity
    plt.figure(figsize=(8, 5))
    plt.plot(future_times, future_humidity, marker='s', color='green')
    plt.title('Future Humidity Predictions')
    plt.xlabel('Time')
    plt.ylabel('Humidity (%)')
    plt.grid(True)
    plt.tight_layout()
    plt.show()    
    
    
    return future_times, future_temp, current_temp_c
  
    

future_times, future_temp, current_temp = weather_view()

plt.plot(future_times, future_temp, label='Predicted Temp')
plt.axhline(current_temp, color='red', linestyle='--', label='Current Temp')
plt.legend()
plt.title("Temperature Forecast vs Current")
plt.show()




weather_view()


Raw Weather Data: {'city': 'Delhi', 'current_temp': 34, 'feels_like': 33, 'temp_min': 34, 'temp_max': 34, 'humidity': 29, 'description': 'clear sky', 'country': 'IN', 'wind_gust_dir': 145, 'pressure': 1010, 'wind_gust_speed': 3.45}
Mean squared error for rain model:
0.1506849315068493


UnboundLocalError: cannot access local variable 'temp_model' where it is not associated with a value